# FairRoute scaling metrics

Reads simulator CSVs and routing sidecars. Missing runs are reported, never replaced with zeros.

In [ ]:
from pathlib import Path
import csv, json, statistics
ROOT=Path.cwd(); RUNS=ROOT/'outputs/fairroute/scaling'; rows=[]
def pct(values,p):
  if not values: return None
  values=sorted(values); return values[min(len(values)-1,int((len(values)-1)*p))]
for path in sorted(RUNS.glob('*.csv')):
  with path.open(newline='') as handle: data=list(csv.DictReader(handle))
  if not data: continue
  complete=[r for r in data if int(float(r.get('output',0)))>0 and int(float(r.get('end_time',-1)))>=0]
  lat=[float(r['latency']) for r in complete]
  sidecar=path.with_suffix(path.suffix+'.routing.json')
  diagnostics=json.loads(sidecar.read_text()) if sidecar.exists() else {}
  rows.append({'run_id':path.stem,'requests':len(data),'completed':len(complete),'completion_fraction':len(complete)/len(data),'latency_p50':pct(lat,.5),'latency_p95':pct(lat,.95),'latency_p99':pct(lat,.99),'decision_p50_ns':diagnostics.get('decision_ns',{}).get('p50'),'decision_p99_ns':diagnostics.get('decision_ns',{}).get('p99')})
print(f'computed metrics for {len(rows)} result files')

In [ ]:
import csv
out=ROOT/'outputs/fairroute/scaling/metrics.csv'
if rows:
  with out.open('w',newline='') as handle:
    writer=csv.DictWriter(handle,fieldnames=rows[0]); writer.writeheader(); writer.writerows(rows)
print(out)